In [0]:
from pyspark.sql import functions as F
products=spark.table('chat_bot.bronze.products')\
        .dropDuplicates(['product_id'])\
        .filter(F.col('price')>0)\
        .withColumn('product_name',F.trim(F.col('product_name')))\
        .write.mode('overwrite').saveAsTable('chat_bot.silver.products')    
customers=spark.table('chat_bot.bronze.customers')\
          .dropDuplicates(['customer_id'])\
          .withColumn('customer_name',F.trim(F.col("customer_name")))\
          .write.mode("overwrite").saveAsTable("chat_bot.silver.customers")    

sales = (spark.table("chat_bot.bronze.sales")
    .dropDuplicates(["sale_id"])
    .filter((F.col("quantity") > 0) & (F.col("total_amount") > 0))
    .withColumn("sale_date", F.to_date("sale_date"))
    .withColumn("sale_month", F.date_format("sale_date", "yyyy-MM")))
sales.write.mode("overwrite").saveAsTable("chat_bot.silver.sales")


for t in ["products", "customers", "sales"]:
    print(t, spark.table(f"chat_bot.silver.{t}").count())

                  
